In [ ]:
# --- portable setup (added 2026-09-12 when the project moved to GitHub) ---------------------
# All paths below resolve from HME_ROOT, the repository root.  On Colab: mount Drive and point
# HME_ROOT at your clone.  Locally: run jupyter from the repo, or export HME_ROOT=/path/to/repo.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    HME_ROOT = os.environ.get('HME_ROOT', '/content/drive/MyDrive/hyperbolic-icd10')   # <-- edit
except ImportError:
    HME_ROOT = os.environ.get('HME_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
assert os.path.isdir(os.path.join(HME_ROOT, 'data')), f'HME_ROOT={HME_ROOT!r} is not the repo root'
print('HME_ROOT =', HME_ROOT)


# Phase 2: BaselinesTrains and evaluates baseline embedding methods on ICD-10-CM.
## Session start (run these every session)
1. Mount drive
2. Imports and constants
3. Load processed data
4. Build integer index mappings
## Shared utilities (run once per session)
5. Negative sampling functions
6. Loss function
7. Training loop functions
8. Evaluation function
## Baselines- [x] Euclidean d=10 — COMPLETED
- [ ] Poincaré K=−1 d=10
- [ ] Learnable K d=10
- [ ] Mixed-curvature product spaces
## Multi-dimension sweep (after baselines work)
- Each baseline at d=5, d=10, d=50

**Prerequisite:** Phase 1 must have run successfully, producing `icd10_tree_with_features.pkl`.

## Session Start: Mount Drive

Mounted at /content/drive


In [27]:
# Init sweep_results if it doesn't exist
if 'sweep_results' not in globals():
    sweep_results = {}

In [28]:
!pip install geoopt --quiet
import geoopt
print(f"geoopt version: {geoopt.__version__}")

geoopt version: 0.5.1


## Session Start: Imports and Constants

In [29]:
import gc
import pickle
import os
import json
import numpy as np
import torch
import torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
from datetime import datetime

PROJECT_ROOT = HME_ROOT
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'phase2_baselines')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models', 'phase2_baselines')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# GPU check
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
  print(f"GPU: {torch.cuda.get_device_name(0)}")
  print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
  print("WARNING: Running on CPU. Switch runtime to GPU (Runtime > Change runtime type > GPU > A100).")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 42.4 GB


In [30]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## Session Start: Load Processed Data

In [ ]:
PICKLE_PATH = HME_ROOT + '/data/processed/icd10_tree_with_features.pkl'

with open(PICKLE_PATH, 'rb') as f:
    data = pickle.load(f)

nodes = data['nodes']
edges = data['edges']
features = data['features']
children_of = data['children_of']
parent_of = data['parent_of']
metadata = data['metadata']

print(f"✓ Loaded {len(nodes)} nodes, {len(edges)} edges")
print(f"  Internal nodes: {sum(1 for f in features.values() if f['is_internal'])}")
print(f"  Leaves: {sum(1 for f in features.values() if f['is_leaf'])}")
print(f"  Max depth: {metadata['max_depth']}")
print(f"  Source: ICD-10-CM {metadata['icd10cm_version']}")

✓ Loaded 46817 nodes, 46816 edges
  Internal nodes: 10775
  Leaves: 36042
  Max depth: 7
  Source: ICD-10-CM FY2025


## Session Start: Build Integer Index Mappings

In [ ]:
# Map each node code to an integer index
# PyTorch embeddings work with integer indices, not string codes
code_to_idx = {code: i for i, code in enumerate(sorted(nodes.keys()))}
idx_to_code = {i: code for code, i in code_to_idx.items()}
N = len(code_to_idx)

print(f"Created index for {N} nodes")
print(f"\nExample mappings:")
for code in ['ROOT', 'CH9', 'SEC_I30-I52', 'I50']:
    if code in code_to_idx:
        print(f"  '{code}' -> index {code_to_idx[code]}")

# Convert edges to integer pairs
edges_idx = [(code_to_idx[p], code_to_idx[c]) for p, c in edges]
print(f"\nConverted {len(edges_idx)} edges to integer pairs")

# Build a set of all directly-connected pairs (both directions)
# Used to exclude actual edges when picking negative samples
connected = set()
for p, c in edges_idx:
    connected.add((p, c))
    connected.add((c, p))

print(f"Connected pairs (both directions): {len(connected)}")

Created index for 46817 nodes

Example mappings:
  'ROOT' -> index 27818
  'CH9' -> index 2863
  'I50' -> index 11339

Converted 46816 edges to integer pairs
Connected pairs (both directions): 93632


## Shared Utility: Single-Anchor Negative Sampling

In [ ]:
def sample_negatives(anchor_idx, num_negatives, N, connected, max_attempts=100):
    """
    Sample K random node indices that are not directly connected to the anchor.

    Parameters
    ----------
    anchor_idx : int
        The integer index of the anchor node we're sampling negatives for.
    num_negatives : int
        How many negative samples to generate (typically 10).
    N : int
        Total number of nodes in the ontology (46,817 for our ICD-10 tree).
    connected : set of (int, int) tuples
        Set of all directly-connected pairs, both directions.
        Used to exclude actual edges from negative candidates.
    max_attempts : int
        Safety limit on how many random draws we'll make before giving up.
        Set high enough that rejection is essentially never an issue.

    Returns
    -------
    negatives : list of int
        K integer indices, none equal to anchor_idx, none directly connected to anchor.
    """
    negatives = []
    attempts = 0

    while len(negatives) < num_negatives and attempts < max_attempts:
        # Draw a random candidate
        candidate = np.random.randint(0, N)
        attempts += 1

        # Skip if it's the anchor itself
        if candidate == anchor_idx:
            continue

        # Skip if it's directly connected to the anchor
        if (anchor_idx, candidate) in connected:
            continue

        # Skip if already in our list (avoid duplicates)
        if candidate in negatives:
            continue

        # Accept it
        negatives.append(candidate)

    if len(negatives) < num_negatives:
        raise RuntimeError(f"Could not find {num_negatives} valid negatives after {max_attempts} attempts")

    return negatives

In [ ]:
# Test the negative sampler on a few anchors
test_anchors = ['I50', 'ROOT', 'CH9']

for anchor_code in test_anchors:
    if anchor_code not in code_to_idx:
        print(f"Skipping {anchor_code} (not in index)")
        continue

    anchor_idx = code_to_idx[anchor_code]
    negatives = sample_negatives(anchor_idx, num_negatives=10, N=N, connected=connected)

    print(f"\nAnchor: {anchor_code} (idx {anchor_idx})")
    print(f"  10 negative samples (as codes):")
    for neg_idx in negatives:
        neg_code = idx_to_code[neg_idx]
        neg_desc = nodes[neg_code]['description'][:50] if nodes[neg_code]['description'] else '(no desc)'
        print(f"    idx {neg_idx}: {neg_code} - {neg_desc}")


Anchor: I50 (idx 11339)
  10 negative samples (as codes):
    idx 4664: E74.81 - Disorders of glucose transport, not elsewhere clas
    idx 17867: M24.47 - Recurrent dislocation, ankle, foot and toes
    idx 46344: Z86.011 - Personal history of benign neoplasm of the brain
    idx 40183: T63.441 - Toxic effect of venom of bees, accidental (uninten
    idx 16220: M08.031 - Unspecified juvenile rheumatoid arthritis, right w
    idx 2951: D04.62 - Carcinoma in situ of skin of left upper limb, incl
    idx 43038: V95.41 - Spacecraft crash injuring occupant
    idx 10613: H73.829 - Atrophic nonflaccid tympanic membrane, unspecified
    idx 23537: N99.528 - Other complication of incontinent external stoma o
    idx 2048: C71.6 - Malignant neoplasm of cerebellum

Anchor: ROOT (idx 27818)
  10 negative samples (as codes):
    idx 32707: S62.134 - Nondisplaced fracture of capitate [os magnum] bone
    idx 37513: T23.569 - Corrosion of first degree of back of unspecified h
    idx 4965: F03.A4 

## Shared Utility: Batched Negative Sampling

In [ ]:
def sample_negatives_batched(anchor_indices, num_negatives, N, connected, max_attempts=100):
    """
    Sample negatives for a batch of anchors all at once.

    Parameters
    ----------
    anchor_indices : array-like of int
        Integer indices of the anchor nodes. Shape: [batch_size]
    num_negatives : int
        How many negatives per anchor (typically 10).
    N : int
        Total number of nodes.
    connected : set of (int, int) tuples
        Directly-connected pairs in both directions.
    max_attempts : int
        Safety limit.

    Returns
    -------
    negatives : np.ndarray of shape [batch_size, num_negatives]
        Integer indices, one row per anchor.
    """
    batch_size = len(anchor_indices)

    # Pre-allocate output array
    negatives = np.zeros((batch_size, num_negatives), dtype=np.int64)

    # For each anchor, fill in its row
    for i, anchor_idx in enumerate(anchor_indices):
        row_negatives = []
        attempts = 0

        while len(row_negatives) < num_negatives and attempts < max_attempts:
            candidate = np.random.randint(0, N)
            attempts += 1

            if candidate == anchor_idx:
                continue
            if (anchor_idx, candidate) in connected:
                continue
            if candidate in row_negatives:
                continue

            row_negatives.append(candidate)

        if len(row_negatives) < num_negatives:
            raise RuntimeError(f"Could not find {num_negatives} valid negatives for anchor {anchor_idx}")

        negatives[i] = row_negatives

    return negatives

In [ ]:
# Test batched sampling on a small batch of anchors
test_anchor_codes = ['I50', 'I21', 'E11.9', 'J45.909']
test_anchor_indices = [code_to_idx[c] for c in test_anchor_codes if c in code_to_idx]

print(f"Sampling negatives for {len(test_anchor_indices)} anchors at once...")
negs = sample_negatives_batched(test_anchor_indices, num_negatives=10, N=N, connected=connected)

print(f"\nOutput shape: {negs.shape}")  # should be [4, 10]
print(f"Dtype: {negs.dtype}")            # should be int64

print(f"\nFirst row of negatives (for anchor {test_anchor_codes[0]}):")
for neg_idx in negs[0]:
    neg_code = idx_to_code[neg_idx]
    print(f"  idx {neg_idx}: {neg_code}")

Sampling negatives for 4 anchors at once...

Output shape: (4, 10)
Dtype: int64

First row of negatives (for anchor I50):
  idx 21794: M89.311
  idx 43307: W23
  idx 18671: M43.8X3
  idx 36083: S93.62
  idx 41838: V25.41
  idx 6241: G40.901
  idx 24715: O44.13
  idx 15771: M05.032
  idx 31435: S51.841
  idx 46400: Z87.441


## Shared Utility: Loss Function (Nickel-Kiela Negative-Sampling Softmax)

In [ ]:
def compute_loss_with_hierarchy(
    anchor_emb, positive_emb, negative_embs, distance_fn,
    margin=0.05, lambda_h=1.0,
):
    """Nickel-Kiela softmax loss + hierarchy regularization."""
    # Standard NK loss
    pos_dist = distance_fn(anchor_emb, positive_emb)
    neg_dist = distance_fn(anchor_emb.unsqueeze(1), negative_embs)
    all_dist = torch.cat([pos_dist.unsqueeze(1), neg_dist], dim=1)
    logsumexp_term = torch.logsumexp(-all_dist, dim=1)
    nk_per_anchor = pos_dist + logsumexp_term

    # Hierarchy regularization: penalize parent norm > child norm - margin
    anchor_norms = torch.norm(anchor_emb, dim=-1)
    positive_norms = torch.norm(positive_emb, dim=-1)
    hierarchy_violation = torch.relu(anchor_norms - positive_norms + margin)

    return nk_per_anchor.mean() + lambda_h * hierarchy_violation.mean()

##Poincare FIXED & LEARNABLE K Utility: Per-Epoch Training Loop

In [ ]:
def train_one_epoch_with_freeze_and_hierarchy(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0,
    freeze_indices=None,
):
    """One training epoch with optional gradient-zero freeze on specific rows."""
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)
        positive_emb = model(positive_t)
        negative_embs = model(negative_t)

        loss = compute_loss_with_hierarchy(
            anchor_emb, positive_emb, negative_embs,
            model.distance,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        # Zero out gradient rows for frozen indices (e.g., ROOT)
        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)

## Euclidean Utility: Per-Epoch Training Loop

In [ ]:
def train_one_epoch(model, edges_idx, optimizer, batch_size, num_negatives, N, connected, device):
    """
    Train for one epoch: pass through all positive edges once, in random order.

    Parameters
    ----------
    model : nn.Module
        The embedding model (e.g., EuclideanEmbedding).
    edges_idx : list of (int, int) tuples
        The full set of positive parent-child edges, as integer index pairs.
    optimizer : torch.optim.Optimizer
        Optimizer instance, e.g., Adam.
    batch_size : int
        Number of positive pairs per batch.
    num_negatives : int
        How many negatives to sample per positive (typically 10).
    N : int
        Total number of nodes (for negative sampling bounds).
    connected : set of (int, int) tuples
        Used for negative-sample rejection.
    device : torch.device
        'cpu' or 'cuda'. Where to do the computation.

    Returns
    -------
    avg_loss : float
        Mean loss across all batches in this epoch.
    """
    model.train()  # Set the model to training mode (vs. evaluation mode)

    # Shuffle the edges so different batches see different pairs
    # We make a copy to avoid modifying the caller's list
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    num_batches = 0

    # Wrap the batch iteration in tqdm for a progress bar
    batch_starts = range(0, len(shuffled_edges), batch_size)

    for batch_start in tqdm(batch_starts, desc="Training", leave=False):
        # Slice out this batch's edges
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        # Separate parents and children
        # In Nickel-Kiela style training, the parent is the "anchor"
        # and the child is the "positive"
        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)

        # Sample negatives for each anchor
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected
        )

        # Convert to PyTorch tensors and move to device
        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        # Forward pass: look up embeddings
        anchor_emb = model(anchor_t)        # [B, D]
        positive_emb = model(positive_t)    # [B, D]
        negative_embs = model(negative_t)   # [B, K, D]

        # Compute loss
        loss = compute_loss_with_hierarchy(anchor_emb, positive_emb, negative_embs, model.distance)

        # Backward pass: compute gradients
        optimizer.zero_grad()  # Clear any leftover gradients from previous batch
        loss.backward()        # Compute gradients via backprop
        optimizer.step()       # Apply the update

        total_loss += loss.item() * actual_batch_size
        num_batches += 1

    avg_loss = total_loss / len(shuffled_edges)
    return avg_loss

## Shared Utility: Evaluation Function (Mean Rank, MAP, MRR)

In [ ]:
def evaluate_reconstruction(model, edges_idx, N, device, sample_size=None, batch_size=512):
    """
    Compute mean rank and MAP for reconstruction quality.

    For each parent-child edge (u, v):
      - Compute distance from u to every node in the tree
      - Find the rank of v in that sorted list of distances
      - Record the rank

    Mean rank = average of all recorded ranks.

    For MAP, we group by anchor u and compute average precision per anchor.

    Parameters
    ----------
    model : nn.Module
        The trained embedding model.
    edges_idx : list of (int, int) tuples
        All positive edges in the tree.
    N : int
        Total number of nodes.
    device : torch.device
        Where the model lives.
    sample_size : int or None
        If not None, evaluate on a random subset of edges (for speed during dev).
        If None, evaluate on all edges.
    batch_size : int
        How many edges to process at once on the GPU.

    Returns
    -------
    metrics : dict
        {'mean_rank': float, 'median_rank': float, 'MAP': float, 'mrr': float}
    """
    model.eval()  # Switch to evaluation mode (disables dropout etc.)

    # Optionally sample edges for faster evaluation during development
    if sample_size is not None and sample_size < len(edges_idx):
        eval_edges = list(edges_idx)
        np.random.shuffle(eval_edges)
        eval_edges = eval_edges[:sample_size]
    else:
        eval_edges = edges_idx

    # Pre-compute ALL embeddings once. Shape: [N, D]
    # This is the key efficiency: instead of looking up embeddings per query,
    # we have them all in one tensor and use matrix operations.
    with torch.no_grad():
        all_indices = torch.arange(N, device=device)
        all_embeddings = model(all_indices)  # [N, D]

    # Build a lookup: for each anchor, what are all its true positive neighbors?
    # This is needed for MAP, where one anchor can have multiple correct answers.
    positives_of = {}
    for u, v in edges_idx:
        # Both directions are valid "true neighbors" of each other
        positives_of.setdefault(u, set()).add(v)
        positives_of.setdefault(v, set()).add(u)

    ranks = []
    reciprocal_ranks = []
    average_precisions = []

    # Process eval_edges in batches
    with torch.no_grad():
        for batch_start in tqdm(range(0, len(eval_edges), batch_size), desc="Evaluating"):
            batch = eval_edges[batch_start : batch_start + batch_size]

            anchor_indices = torch.tensor([u for u, v in batch], device=device)  # [B]
            target_indices = torch.tensor([v for u, v in batch], device=device)  # [B]

            # Get anchor embeddings: [B, D]
            anchor_emb = model(anchor_indices)

            # Compute distance from each anchor to ALL nodes
            # anchor_emb: [B, D] -> [B, 1, D] via unsqueeze
            # all_embeddings: [N, D] -> [1, N, D] via unsqueeze
            # Broadcasting gives distances of shape [B, N]
            distances = model.distance(anchor_emb.unsqueeze(1), all_embeddings.unsqueeze(0))

            # For each anchor in the batch, compute rank of its target
            for i in range(len(batch)):
                u = anchor_indices[i].item()
                v = target_indices[i].item()

                # Get distances from this anchor to all nodes
                dists = distances[i]  # [N]

                # We want to know: how many nodes are closer to u than v is?
                # That's the rank of v (with 1-based indexing where rank 1 = closest).
                # Exclude u itself from the ranking (a node isn't its own neighbor).
                # Also exclude v itself from the comparison (we don't compare v to itself).

                target_distance = dists[v].item()
                # Count how many other nodes are at distance <= target_distance
                # We exclude u from this comparison
                mask = torch.ones(N, dtype=torch.bool, device=device)
                mask[u] = False  # exclude anchor itself

                closer_count = ((dists <= target_distance) & mask).sum().item()
                # closer_count includes v itself, so rank = closer_count
                # (rank 1 means only v is at that distance or closer)
                rank = closer_count

                ranks.append(rank)
                reciprocal_ranks.append(1.0 / rank)

                # For MAP, compute average precision over all of u's true neighbors
                u_positives = positives_of.get(u, set()) - {u}
                if len(u_positives) > 0:
                    # Sort all nodes by distance from u (excluding u itself)
                    sorted_indices = dists.argsort()  # ascending distance
                    sorted_indices = sorted_indices[sorted_indices != u]  # remove u

                    # Walk through the sorted list; compute precision at each true positive
                    num_correct = 0
                    precisions = []
                    for k, idx in enumerate(sorted_indices.cpu().numpy(), 1):
                        if idx in u_positives:
                            num_correct += 1
                            precisions.append(num_correct / k)
                            if num_correct == len(u_positives):
                                break

                    if precisions:
                        average_precisions.append(np.mean(precisions))

    return {
        'mean_rank': float(np.mean(ranks)),
        'median_rank': float(np.median(ranks)),
        'MAP': float(np.mean(average_precisions)) if average_precisions else 0.0,
        'mrr': float(np.mean(reciprocal_ranks)),
        'num_evaluated': len(ranks),
    }

# Baseline 1: Euclidean Embedding (d=10) — COMPLETED

## Define EuclideanEmbedding Model Class

In [ ]:
class EuclideanEmbedding(nn.Module):
    """
    A simple Euclidean embedding model.

    Stores N embedding vectors of dimension d. Given a batch of integer indices,
    returns the corresponding embedding vectors. Distance between two embeddingswwww
    is just standard Euclidean distance.

    Parameters
    ----------
    num_nodes : int
        Number of distinct nodes to embed (46,817 for our ICD-10 tree).
    dim : int
        Dimensionality of each embedding vector (e.g., 10).
    init_scale : float
        Initial embedding values are drawn from uniform(-init_scale, +init_scale).
        Small initial values mean embeddings start near origin and spread out during training.
    """

    def __init__(self, num_nodes, dim, init_scale=0.001):
        super().__init__()

        # The embedding table: one row per node, each row a d-dimensional vector
        # nn.Embedding is PyTorch's standard way to store learnable lookup tables
        self.embeddings = nn.Embedding(num_nodes, dim)

        # Initialize all embeddings to small random values near zero
        # This breaks symmetry (all nodes starting identical) but stays small enough
        # that initial distances are well-behaved
        nn.init.uniform_(self.embeddings.weight, -init_scale, init_scale)

        # Save dim for reference
        self.dim = dim
        self.num_nodes = num_nodes

    def forward(self, indices):
        """
        Look up embeddings for a batch of indices.

        Parameters
        ----------
        indices : torch.LongTensor
            Integer indices of shape [...] (any shape). Each value in [0, num_nodes).

        Returns
        -------
        embeddings : torch.FloatTensor
            Embedding vectors with one extra trailing dimension.
            If indices has shape [B, K], output has shape [B, K, dim].
        """
        return self.embeddings(indices)

    def distance(self, u, v):
        """
        Euclidean distance between two batches of embedding vectors.

        Parameters
        ----------
        u : torch.FloatTensor of shape [..., dim]
        v : torch.FloatTensor of shape [..., dim]
            u and v must have shapes that broadcast together.

        Returns
        -------
        d : torch.FloatTensor of shape [...]
            Euclidean distance, one per pair. The trailing dim axis is collapsed.
        """
        # Element-wise subtraction, then take L2 norm along the last dimension
        # ‖u - v‖_2 = sqrt(sum_i (u_i - v_i)^2)
        return torch.norm(u - v, p=2, dim=-1)

## Train Euclidean d=10

In [ ]:
def train_euclidean_baseline(edges_idx, N, connected, dim=10, num_epochs=50,
                              batch_size=1024, num_negatives=10, learning_rate=0.1,
                              device=None):
    """
    Full training driver for the Euclidean baseline.

    Parameters
    ----------
    edges_idx : list of (int, int) tuples
        Parent-child edges as integer pairs.
    N : int
        Total number of nodes.
    connected : set
        For negative-sample rejection.
    dim : int
        Embedding dimension.
    num_epochs : int
        How many full passes through the edges.
    batch_size : int
        Positive pairs per batch.
    num_negatives : int
        Negatives per positive.
    learning_rate : float
        Adam learning rate.
    device : torch.device or None
        Device to train on. If None, auto-detects CUDA.

    Returns
    -------
    model : EuclideanEmbedding
        The trained model.
    loss_history : list of float
        Mean loss per epoch.
    """
    # Auto-select device if not specified
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on device: {device}")

    # Create model and move it to device
    model = EuclideanEmbedding(num_nodes=N, dim=dim)
    model = model.to(device)

    # Create optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Train
    loss_history = []
    for epoch in range(num_epochs):
        avg_loss = train_one_epoch(
            model, edges_idx, optimizer, batch_size, num_negatives,
            N, connected, device
        )
        loss_history.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}: avg loss = {avg_loss:.4f}")

    return model, loss_history

In [ ]:
# Train Euclidean d=10 at K=50, save, evaluate
model_euc_d10_K50, history_d10_K50 = train_euclidean_baseline(
    edges_idx=edges_idx,
    N=N,
    connected=connected,
    dim=10,
    num_negatives=50,
    batch_size=1024,
    learning_rate=0.1,
    num_epochs=50,
    device=device,
)

save_path = os.path.join(MODELS_DIR, 'euclidean_d10_K50.pt')
torch.save({
    'model_state_dict': model_euc_d10_K50.state_dict(),
    'history': history_d10_K50,
    'config': {'dim': 10, 'num_negatives': 50, 'batch_size': 1024,
               'learning_rate': 0.1, 'num_epochs': 50},
}, save_path)
print(f"\nSaved model to {save_path}")

results_d10_K50 = evaluate_reconstruction(
    model_euc_d10_K50, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
print("\nResults:")
for k, v in results_d10_K50.items():
    print(f"  {k}: {v}")

results_path = os.path.join(RESULTS_DIR, 'euclidean_d10_K50_results.json')
with open(results_path, 'w') as f:
    json.dump({k: float(v) if isinstance(v, (np.floating, np.integer)) else v
               for k, v in results_d10_K50.items()}, f, indent=2)
print(f"\nResults saved to {results_path}")

Training on device: cuda


Epoch 1/50: avg loss = 5.1135


Epoch 2/50: avg loss = 3.7940


Epoch 3/50: avg loss = 2.6044


Epoch 4/50: avg loss = 1.7343


...

Epoch 48/50: avg loss = 0.0373


Epoch 49/50: avg loss = 0.0360


Epoch 50/50: avg loss = 0.0356

Saved model to /content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/models/euclidean_d10_K50.pt


Evaluating: 100%|██████████| 183/183 [00:30<00:00,  6.08it/s]


Results:
  mean_rank: 7.070125598086125
  median_rank: 3.0
  MAP: 0.8606489867830377
  mrr: 0.42471767679789724
  num_evaluated: 46816

Results saved to /content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/results/euclidean_d10_K50_results.json


## Inspect Trained Embeddings

In [ ]:
# Pick a few related and unrelated codes

related_codes = ['I50', 'I50.1', 'I50.2', 'I50.3', 'I50.21']  # heart failure family
unrelated_codes = ['I50', 'F32.9', 'S72.001', 'Z00.00']  # heart failure vs unrelated stuff

def show_distances(model, code_list, code_to_idx, idx_to_code, device):
    indices = torch.tensor([code_to_idx[c] for c in code_list]).to(device)
    embeddings = model(indices)  # shape [len(code_list), dim]

    print(f"\nPairwise distances between: {code_list}")
    print(f"{'':15s}", end='')
    for c in code_list:
        print(f"{c[:10]:>12s}", end='')
    print()

    for i, c1 in enumerate(code_list):
        print(f"{c1[:14]:15s}", end='')
        for j, c2 in enumerate(code_list):
            d = model.distance(embeddings[i:i+1], embeddings[j:j+1]).item()
            print(f"{d:>12.4f}", end='')
        print()

device = next(trained_model.parameters()).device
show_distances(trained_model, related_codes, code_to_idx, idx_to_code, device)
show_distances(trained_model, unrelated_codes, code_to_idx, idx_to_code, device)


Pairwise distances between: ['I50', 'I50.1', 'I50.2', 'I50.3', 'I50.21']
                        I50       I50.1       I50.2       I50.3      I50.21
I50                  0.0000      2.4776      3.0454      5.6703      5.7181
I50.1                2.4776      0.0000      3.0050      5.5498      5.8351
I50.2                3.0454      3.0050      0.0000      6.4133      3.3076
I50.3                5.6703      5.5498      6.4133      0.0000      7.9824
I50.21               5.7181      5.8351      3.3076      7.9824      0.0000

Pairwise distances between: ['I50', 'F32.9', 'S72.001', 'Z00.00']
                        I50       F32.9     S72.001      Z00.00
I50                  0.0000     15.8979     10.5299     14.9117
F32.9               15.8979      0.0000     19.9025     13.6711
S72.001             10.5299     19.9025      0.0000     18.8905
Z00.00              14.9117     13.6711     18.8905      0.0000


## Full Evaluation on All Edges

In [ ]:
# Full evaluation - takes a few minutes
print("Running full evaluation on all 46,816 edges...")
full_metrics_euclidean_d10 = evaluate_reconstruction(
    trained_model, edges_idx, N, device,
    sample_size=None,  # use all edges
    batch_size=256
)
print("\nFull Euclidean d=10 baseline results:")
for k, v in full_metrics_euclidean_d10.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Running full evaluation on all 46,816 edges...


Evaluating: 100%|██████████| 183/183 [00:30<00:00,  6.07it/s]


Full Euclidean d=10 baseline results:
  mean_rank: 32.2515
  median_rank: 3.0000
  MAP: 0.7471
  mrr: 0.3903
  num_evaluated: 46816


## Save Trained Model and Results

In [ ]:
# Save the trained model
model_path = os.path.join(MODELS_DIR, 'euclidean_d10.pt')
torch.save({
    'model_state_dict': trained_model.state_dict(),
    'num_nodes': N,
    'dim': 10,
    'method': 'euclidean',
    'final_loss': loss_history[-1],
    'loss_history': loss_history,
    'training_config': {
        'num_epochs': 50,
        'batch_size': 1024,
        'num_negatives': 10,
        'learning_rate': 0.1,
        'init_scale': 0.001,
    },
    'date_trained': datetime.now().isoformat(),
}, model_path)
print(f"Model saved to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1e6:.2f} MB")

# Save evaluation results
results_path = os.path.join(RESULTS_DIR, 'euclidean_d10_results.json')
results_to_save = {
    'method': 'euclidean',
    'dim': 10,
    'metrics': full_metrics_euclidean_d10,
    'training_loss_history': loss_history,
    'date_evaluated': datetime.now().isoformat(),
}
with open(results_path, 'w') as f:
    json.dump(results_to_save, f, indent=2)
print(f"\nResults saved to: {results_path}")

Model saved to: /content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/models/euclidean_d10.pt
File size: 1.88 MB

Results saved to: /content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/results/euclidean_d10_results.json


##Euclidean D=5

In [ ]:


try: del trained_eucl_d5
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Euclidean d=5")
print("=" * 60)
trained_eucl_d5, loss_history_eucl_d5 = train_euclidean_baseline(
    edges_idx=edges_idx, N=N, connected=connected,
    dim=5, num_epochs=50, batch_size=1024,
    num_negatives=10, learning_rate=0.1,
)

print("\nEvaluating Euclidean d=5...")
metrics_eucl_d5 = evaluate_reconstruction(
    trained_eucl_d5, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_eucl_d5.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_eucl_d5.state_dict(),
    'method': 'euclidean', 'dim': 5,
    'metrics': metrics_eucl_d5, 'final_loss': loss_history_eucl_d5[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'euclidean_d5.pt'))

sweep_results['euclidean_d5'] = metrics_eucl_d5

Euclidean d=5
Training on device: cuda


Epoch 1/50: avg loss = 2.8516


Epoch 2/50: avg loss = 2.4858


Epoch 3/50: avg loss = 1.9501


Epoch 4/50: avg loss = 1.4822


...

Epoch 48/50: avg loss = 0.0984


Epoch 49/50: avg loss = 0.0950


Epoch 50/50: avg loss = 0.0945

Evaluating Euclidean d=5...


Evaluating:  57%|█████▋    | 105/183 [00:18<00:13,  5.60it/s]


KeyboardInterrupt: 

##Euclidean D=50

In [ ]:
try: del trained_eucl_d50
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Euclidean d=50")
print("=" * 60)
trained_eucl_d50, loss_history_eucl_d50 = train_euclidean_baseline(
    edges_idx=edges_idx, N=N, connected=connected,
    dim=50, num_epochs=50, batch_size=1024,
    num_negatives=10, learning_rate=0.1,
)

print("\nEvaluating Euclidean d=50...")
metrics_eucl_d50 = evaluate_reconstruction(
    trained_eucl_d50, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_eucl_d50.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_eucl_d50.state_dict(),
    'method': 'euclidean', 'dim': 50,
    'metrics': metrics_eucl_d50, 'final_loss': loss_history_eucl_d50[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'euclidean_d50.pt'))

sweep_results['euclidean_d50'] = metrics_eucl_d50

Euclidean d=50
Training on device: cuda


Epoch 1/50: avg loss = 3.6120


Epoch 2/50: avg loss = 1.9460


Epoch 3/50: avg loss = 0.7574


Epoch 4/50: avg loss = 0.3664


...

Epoch 48/50: avg loss = 0.0740


Epoch 49/50: avg loss = 0.0768


Epoch 50/50: avg loss = 0.0850

Evaluating Euclidean d=50...


Evaluating: 100%|██████████| 183/183 [00:33<00:00,  5.51it/s]


  mean_rank: 109.80707450444292
  median_rank: 3.0
  MAP: 0.900106391940014
  mrr: 0.4170461499720877
  num_evaluated: 46816


# Baseline 2: Poincaré K=−1 (d=10)

##Install geoopt for Poincare ball manifolds, Riemannian gradient descent, Riemannian Adam, Riemannian Optimization

##Define the Poincare Embedding model

In [ ]:
class PoincareEmbedding(nn.Module):
    """
    Poincaré ball embedding model with constant curvature K = -1.

    Stores N embedding vectors that live inside the open unit ball in d dimensions.
    The Poincaré metric makes distances explode near the unit sphere, which gives
    hyperbolic space its exponential volume growth — the property that lets
    low-dimensional embeddings capture tree structure efficiently.
    """

    def __init__(self, num_nodes, dim, init_scale=0.001):
        super().__init__()

        # The Poincaré ball manifold from geoopt.
        # The 'c' parameter is the absolute value of curvature: c=1 means K=-1.
        # Internally, geoopt uses negative-curvature conventions consistent with
        # Nickel-Kiela's original paper.
        self.manifold = geoopt.PoincareBall(c=1.0)

        # The embedding table, but as a ManifoldParameter that geoopt knows
        # is constrained to live on the Poincaré ball manifold. This is what
        # tells the Riemannian optimizer to respect the unit-ball constraint.
        self.embeddings = geoopt.ManifoldParameter(
            torch.empty(num_nodes, dim),
            manifold=self.manifold
        )

        # Initialize embeddings to small random values near the origin.
        # Near-origin initialization is critical for hyperbolic embeddings:
        # the metric becomes singular at the boundary, so starting close to
        # the boundary causes numerical instability.
        # The .data accessor lets us modify the parameter in-place without
        # tracking gradients.
        with torch.no_grad():
            self.embeddings.data.uniform_(-init_scale, init_scale)

        self.dim = dim
        self.num_nodes = num_nodes

    def forward(self, indices):
        """
        Look up embeddings for a batch of indices.

        Parameters
        ----------
        indices : torch.LongTensor of any shape
            Integer indices into the embedding table.

        Returns
        -------
        embeddings : torch.FloatTensor
            One trailing dimension of size self.dim added to the input shape.
        """
        # Index into the embedding table along axis 0
        return self.embeddings[indices]

    def distance(self, u, v):
        """
        Poincaré distance between two batches of embeddings.

        Parameters
        ----------
        u, v : torch.FloatTensor of shape [..., dim]
            Two batches of points in the Poincaré ball.

        Returns
        -------
        d : torch.FloatTensor of shape [...]
            Hyperbolic distance, one per pair.
        """
        # geoopt's manifold object provides the correct distance function.
        # Internally this computes d(u,v) = arcosh(1 + 2||u-v||^2 / [(1-||u||^2)(1-||v||^2)])
        return self.manifold.dist(u, v, keepdim=False)

##Sanity-check the model before training

In [ ]:
# Quick sanity check before training
test_model_p = PoincareEmbedding(num_nodes=N, dim=10).to(device)

print(f"Created PoincareEmbedding with {N} nodes, dim=10")
print(f"Total parameters: {sum(p.numel() for p in test_model_p.parameters())}")

# Look up a few embeddings
test_idx = torch.tensor([0, 100, 1000], device=device)
test_emb = test_model_p(test_idx)
print(f"\nEmbedding lookup:")
print(f"  Input shape: {test_idx.shape}")
print(f"  Output shape: {test_emb.shape}")
print(f"  All norms < 1: {(torch.norm(test_emb, dim=-1) < 1).all().item()}")

# Test distance
u_test = test_model_p(torch.tensor([0, 1], device=device))
v_test = test_model_p(torch.tensor([100, 101], device=device))
d_test = test_model_p.distance(u_test, v_test)
print(f"\nDistance:")
print(f"  u shape: {u_test.shape}, v shape: {v_test.shape}")
print(f"  Distance shape: {d_test.shape}")
print(f"  Distances: {d_test}")

Created PoincareEmbedding with 46817 nodes, dim=10
Total parameters: 468171

Embedding lookup:
  Input shape: torch.Size([3])
  Output shape: torch.Size([3, 10])
  All norms < 1: True

Distance:
  u shape: torch.Size([2, 10]), v shape: torch.Size([2, 10])
  Distance shape: torch.Size([2])
  Distances: tensor([0.0050, 0.0043], device='cuda:0', grad_fn=<MulBackward1>)


##Set up Riemannian Optimization + Freeze Root at origin, don't let it deviate, let others move around.

In [ ]:
def train_one_epoch_with_freeze_and_hierarchy(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0,
    freeze_indices=None,
):
    """train_one_epoch_with_freeze, but using compute_loss_with_hierarchy."""
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)
        positive_emb = model(positive_t)
        negative_embs = model(negative_t)

        loss = compute_loss_with_hierarchy(
            anchor_emb, positive_emb, negative_embs,
            model.distance,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)


def train_poincare_with_freeze_and_hierarchy(
    edges_idx, N, connected, code_to_idx,
    dim=10,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    batch_size=1024,
    num_negatives=50,
    learning_rate=10.0,
    burnin_multiplier=0.01,
    init_scale=0.001,
    margin=0.05,
    lambda_h=1.0,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("=" * 60)
    print(f"🔒+🏛️ ROOT-FREEZE + HIERARCHY Poincaré training")
    print(f"   ROOT idx = {code_to_idx['ROOT']}, pinned for {burnin_epochs + freeze_train_epochs} epochs")
    print(f"   hierarchy: margin={margin}, lambda_h={lambda_h}")
    print(f"   lr={learning_rate}, num_negatives={num_negatives}")
    print("=" * 60)

    model = PoincareEmbedding(num_nodes=N, dim=dim, init_scale=init_scale).to(device)

    root_idx = code_to_idx['ROOT']
    with torch.no_grad():
        model.embeddings.data[root_idx].zero_()

    with torch.no_grad():
        root_norm_at_init = torch.norm(model.embeddings.data[root_idx]).item()
        print(f"After init: ROOT_norm = {root_norm_at_init:.6f}   (must be 0.000000)")

    optimizer = geoopt.optim.RiemannianSGD(model.parameters(), lr=learning_rate)

    loss_history = []
    freeze_until_epoch = burnin_epochs + freeze_train_epochs

    for epoch in range(num_epochs):
        effective_lr = (learning_rate * burnin_multiplier
                        if epoch < burnin_epochs else learning_rate)
        for pg in optimizer.param_groups:
            pg['lr'] = effective_lr

        freeze_indices = [root_idx] if epoch < freeze_until_epoch else None

        avg_loss = train_one_epoch_with_freeze_and_hierarchy(
            model, edges_idx, optimizer,
            batch_size, num_negatives, N, connected, device,
            margin=margin, lambda_h=lambda_h,
            freeze_indices=freeze_indices,
        )
        loss_history.append(avg_loss)

        if epoch < burnin_epochs:
            phase = "burn-in+freeze"
        elif epoch < freeze_until_epoch:
            phase = "FROZEN+hier"
        else:
            phase = "free+hier"

        if (epoch < 3 or epoch == burnin_epochs or epoch == freeze_until_epoch
                or (epoch + 1) % 10 == 0):
            with torch.no_grad():
                norms = torch.norm(model.embeddings.data, dim=-1)
                print(
                    f"Epoch {epoch+1:3d}/{num_epochs} [{phase}, lr={effective_lr:.4f}]: "
                    f"loss={avg_loss:.4f}, "
                    f"ROOT_norm={norms[code_to_idx['ROOT']].item():.4f}, "
                    f"mean_norm={norms.mean().item():.4f}, "
                    f"max_norm={norms.max().item():.4f}"
                )

    return model, loss_history

##Run Training

In [ ]:
import gc
try: del trained_poincare
except NameError: pass
gc.collect()
torch.cuda.empty_cache()

trained_poincare, loss_history_poincare = train_poincare_with_freeze_and_hierarchy(
    edges_idx=edges_idx,
    N=N,
    connected=connected,
    code_to_idx=code_to_idx,
    dim=10,
    margin=0.05,
    lambda_h=1.0,
    device=device,
)

🔒+🏛️ ROOT-FREEZE + HIERARCHY Poincaré training
   ROOT idx = 27818, pinned for 10010 epochs
   hierarchy: margin=0.05, lambda_h=1.0
   lr=10.0, num_negatives=50
After init: ROOT_norm = 0.000000   (must be 0.000000)


Epoch   1/300 [burn-in+freeze, lr=0.1000]: loss=3.9818, ROOT_norm=0.0000, mean_norm=0.0018, max_norm=0.0028


Epoch   2/300 [burn-in+freeze, lr=0.1000]: loss=3.9816, ROOT_norm=0.0000, mean_norm=0.0018, max_norm=0.0028


Epoch   3/300 [burn-in+freeze, lr=0.1000]: loss=3.9814, ROOT_norm=0.0000, mean_norm=0.0018, max_norm=0.0028


Epoch  10/300 [burn-in+freeze, lr=0.1000]: loss=3.9803, ROOT_norm=0.0000, mean_norm=0.0019, max_norm=0.0030


...

Epoch 260/300 [FROZEN+hier, lr=10.0000]: loss=0.8511, ROOT_norm=0.0000, mean_norm=0.8726, max_norm=0.8943


Epoch 270/300 [FROZEN+hier, lr=10.0000]: loss=0.8191, ROOT_norm=0.0000, mean_norm=0.8790, max_norm=0.9000


Epoch 280/300 [FROZEN+hier, lr=10.0000]: loss=0.7909, ROOT_norm=0.0000, mean_norm=0.8848, max_norm=0.9050


Epoch 290/300 [FROZEN+hier, lr=10.0000]: loss=0.7665, ROOT_norm=0.0000, mean_norm=0.8899, max_norm=0.9096


Epoch 300/300 [FROZEN+hier, lr=10.0000]: loss=0.7445, ROOT_norm=0.0000, mean_norm=0.8945, max_norm=0.9137


In [ ]:
# Evaluate the trained Poincaré model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Running full evaluation of Poincaré K=-1 d=10 on all 46,816 edges...")
full_metrics_poincare_d10 = evaluate_reconstruction(
    trained_poincare, edges_idx, N, device,
    sample_size=None,
    batch_size=256
)
print("\nPoincaré K=-1 d=10 baseline results:")
for k, v in full_metrics_poincare_d10.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Running full evaluation of Poincaré K=-1 d=10 on all 46,816 edges...


Evaluating: 100%|██████████| 183/183 [00:49<00:00,  3.73it/s]


Poincaré K=-1 d=10 baseline results:
  mean_rank: 494.3615
  median_rank: 3.0000
  MAP: 0.7400
  mrr: 0.4086
  num_evaluated: 46816


In [ ]:
import os, torch
from datetime import datetime
MODELS_DIR = HME_ROOT + '/models/phase2_baselines'

torch.save({
    'model_state_dict': trained_poincare.state_dict(),
    'method': 'poincare_K-1_freeze_hier',
    'config': {'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 50,
               'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300},
    'final_loss': 0.7484,
    'metrics': {'mean_rank': 497.19, 'median_rank': 3.0, 'MAP': 0.7408, 'mrr': 0.4097},
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'poincare_d10_freeze_hier.pt'))
print("Saved freeze+hier model.")

Saved freeze+hier model.


In [ ]:
import numpy as np
import torch

device = next(trained_poincare.parameters()).device

# Pull all embeddings and norms once for efficiency
with torch.no_grad():
    all_indices = torch.arange(N, device=device)
    all_embeddings = trained_poincare(all_indices)
    all_norms = torch.norm(all_embeddings, dim=-1).cpu().numpy()

# ============================================================
# Diagnostic 1: Norm distribution by tree depth
# ============================================================
print("=" * 75)
print("DIAGNOSTIC 1: Norm distribution by tree depth")
print("=" * 75)

nodes_by_depth = {}
for code, idx in code_to_idx.items():
    d = features[code]['depth']
    nodes_by_depth.setdefault(d, []).append(idx)

print(f"\n{'Depth':>5} | {'Count':>6} | {'Mean':>7} | {'Min':>7} | {'Median':>7} | {'Max':>7} | {'Std':>7}")
print("-" * 70)
for d in sorted(nodes_by_depth.keys()):
    indices = nodes_by_depth[d]
    norms_d = all_norms[indices]
    print(f"{d:>5d} | {len(indices):>6d} | "
          f"{norms_d.mean():.4f} | {norms_d.min():.4f} | "
          f"{np.median(norms_d):.4f} | {norms_d.max():.4f} | "
          f"{norms_d.std():.4f}")

# ============================================================
# Diagnostic 2: Lineage walk
# ============================================================
print("\n" + "=" * 75)
print("DIAGNOSTIC 2: Walk down a tree lineage (heart-failure path)")
print("=" * 75)

lineage = ['ROOT', 'CH9', 'SEC_I30-I52', 'I50', 'I50.1']
for code in lineage:
    if code not in code_to_idx:
        print(f"\n  {code}: NOT FOUND in index")
        continue
    idx = code_to_idx[code]
    depth = features[code]['depth']
    desc = nodes[code].get('description', '') or ''
    if len(desc) > 60:
        desc = desc[:57] + "..."
    num_children = len(children_of.get(code, []))
    print(f"\n  {code}  (depth {depth})")
    print(f"    norm     = {all_norms[idx]:.4f}")
    print(f"    children = {num_children}")
    print(f"    parent   = {parent_of.get(code, None)}")
    if desc:
        print(f"    desc     = {desc}")

# ============================================================
# Diagnostic 3: Nearest neighbors of a specific node
# ============================================================
print("\n" + "=" * 75)
print("DIAGNOSTIC 3: 15 nearest neighbors of I50 (Heart failure)")
print("=" * 75)

def show_nearest_neighbors(code, k=15):
    if code not in code_to_idx:
        print(f"{code} not found")
        return
    idx = code_to_idx[code]

    with torch.no_grad():
        anchor_emb = trained_poincare(torch.tensor([idx], device=device))
        distances = trained_poincare.distance(
            anchor_emb.unsqueeze(1),
            all_embeddings.unsqueeze(0)
        ).squeeze(0).cpu().numpy()

    parent = parent_of.get(code, None)
    children = set(children_of.get(code, []))
    siblings = (set(children_of.get(parent, [])) - {code}) if parent else set()

    print(f"\n  Query: {code}  (norm={all_norms[idx]:.4f}, depth={features[code]['depth']})")
    print(f"  Parent: {parent} | direct children: {len(children)} | siblings: {len(siblings)}")

    sorted_idx = np.argsort(distances)
    sorted_idx = sorted_idx[sorted_idx != idx][:k]

    print(f"\n  {'Rank':>4} | {'Code':<20s} | {'Norm':>7} | {'Dist':>7} | Relation")
    print("  " + "-" * 70)
    for rank, n_idx in enumerate(sorted_idx, 1):
        n_code = idx_to_code[n_idx]
        if n_code == parent:
            rel = "PARENT (should be near top)"
        elif n_code in children:
            rel = "child (should be near top)"
        elif n_code in siblings:
            rel = "sibling"
        else:
            rel = "other"
        print(f"  {rank:>4d} | {n_code:<20s} | {all_norms[n_idx]:.4f} | {distances[n_idx]:.4f} | {rel}")

show_nearest_neighbors('I50', k=15)

DIAGNOSTIC 1: Norm distribution by tree depth

Depth |  Count |    Mean |     Min |  Median |     Max |     Std
----------------------------------------------------------------------
    0 |      1 | 0.0000 | 0.0000 | 0.0000 | 0.0000 | 0.0000
    1 |     22 | 0.7800 | 0.7312 | 0.7837 | 0.8518 | 0.0259
    2 |    296 | 0.8386 | 0.7419 | 0.8450 | 0.9032 | 0.0319
    3 |   1917 | 0.8801 | 0.7739 | 0.8915 | 0.9123 | 0.0279
    4 |  10070 | 0.8932 | 0.7876 | 0.8981 | 0.9123 | 0.0153
    5 |  14483 | 0.8956 | 0.8344 | 0.8976 | 0.9135 | 0.0102
    6 |  19936 | 0.8966 | 0.8597 | 0.8981 | 0.9136 | 0.0085
    7 |     92 | 0.8985 | 0.8869 | 0.8977 | 0.9137 | 0.0080

DIAGNOSTIC 2: Walk down a tree lineage (heart-failure path)

  ROOT  (depth 0)
    norm     = 0.0000
    children = 22
    parent   = None
    desc     = ICD-10-CM root

  CH9  (depth 1)
    norm     = 0.7385
    children = 10
    parent   = ROOT
    desc     = Diseases of the circulatory system (I00-I99)

  SEC_I30-I52: NOT FOUND in 

In [ ]:
import os, torch
from datetime import datetime

MODELS_DIR = HME_ROOT + '/models/phase2_baselines'
RESULTS_DIR = HME_ROOT + '/results/phase2_baselines'

# Save Poincaré d=10 (attempt 11) if not already saved
d10_path = os.path.join(MODELS_DIR, 'poincare_d10_permanent_freeze_hier.pt')
if not os.path.exists(d10_path):
    torch.save({
        'model_state_dict': trained_poincare.state_dict(),
        'method': 'poincare_K-1_permanent_freeze_hier',
        'dim': 10,
        'config': {
            'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 10000,
            'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300,
            'num_negatives': 50, 'batch_size': 1024,
        },
        'metrics': {'mean_rank': 497.3826, 'median_rank': 3.0,
                    'MAP': 0.7399, 'mrr': 0.4081, 'num_evaluated': 46816},
        'date_trained': datetime.now().isoformat(),
    }, d10_path)
    print(f"Saved Poincaré d=10 to {d10_path}")
else:
    print(f"Poincaré d=10 already saved.")

# Recover the d=5 Euclidean metrics that were computed before the error
sweep_results['euclidean_d5'] = metrics_eucl_d5
print(f"Added euclidean_d5 to sweep_results")
print(f"Current keys: {list(sweep_results.keys())}")
print(f"\nEuclidean d=5 metrics: MAP={metrics_eucl_d5['MAP']:.4f}, "
      f"mean_rank={metrics_eucl_d5['mean_rank']:.2f}")

Poincaré d=10 already saved.
Added euclidean_d5 to sweep_results
Current keys: ['euclidean_d5', 'euclidean_d10', 'poincare_d10', 'euclidean_d50']

Euclidean d=5 metrics: MAP=0.2727, mean_rank=88.81


##Poincare D=5

In [ ]:
try: del trained_poincare_d5
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Poincaré d=5 (RSGD + permanent ROOT freeze + hierarchy)")
print("=" * 60)
trained_poincare_d5, loss_history_poincare_d5 = train_poincare_with_freeze_and_hierarchy(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    dim=5,
    num_epochs=300, burnin_epochs=10, freeze_train_epochs=10000,  # permanent
    learning_rate=10.0, margin=0.05, lambda_h=1.0,
    device=device,
)

print("\nEvaluating Poincaré d=5...")
metrics_poincare_d5 = evaluate_reconstruction(
    trained_poincare_d5, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_poincare_d5.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_poincare_d5.state_dict(),
    'method': 'poincare_K-1_permanent_freeze_hier', 'dim': 5,
    'config': {
        'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300,
        'num_negatives': 50, 'batch_size': 1024,
    },
    'metrics': metrics_poincare_d5,
    'final_loss': loss_history_poincare_d5[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'poincare_d5_permanent_freeze_hier.pt'))

sweep_results['poincare_d5'] = metrics_poincare_d5

Poincaré d=5 (RSGD + permanent ROOT freeze + hierarchy)
🔒+🏛️ ROOT-FREEZE + HIERARCHY Poincaré training
   ROOT idx = 27818, pinned for 10010 epochs
   hierarchy: margin=0.05, lambda_h=1.0
   lr=10.0, num_negatives=50
After init: ROOT_norm = 0.000000   (must be 0.000000)


Epoch   1/300 [burn-in+freeze, lr=0.1000]: loss=3.9818, ROOT_norm=0.0000, mean_norm=0.0013, max_norm=0.0022


Epoch   2/300 [burn-in+freeze, lr=0.1000]: loss=3.9816, ROOT_norm=0.0000, mean_norm=0.0013, max_norm=0.0022


Epoch   3/300 [burn-in+freeze, lr=0.1000]: loss=3.9814, ROOT_norm=0.0000, mean_norm=0.0013, max_norm=0.0022


Epoch  10/300 [burn-in+freeze, lr=0.1000]: loss=3.9805, ROOT_norm=0.0000, mean_norm=0.0014, max_norm=0.0025


...

Epoch 280/300 [FROZEN+hier, lr=10.0000]: loss=0.8634, ROOT_norm=0.0000, mean_norm=0.8839, max_norm=0.9106


Epoch 290/300 [FROZEN+hier, lr=10.0000]: loss=0.8313, ROOT_norm=0.0000, mean_norm=0.8898, max_norm=0.9150


Epoch 300/300 [FROZEN+hier, lr=10.0000]: loss=0.8038, ROOT_norm=0.0000, mean_norm=0.8950, max_norm=0.9192

Evaluating Poincaré d=5...


Evaluating: 100%|██████████| 183/183 [01:06<00:00,  2.74it/s]


  mean_rank: 989.374765037594
  median_rank: 3.0
  MAP: 0.6827185274146321
  mrr: 0.391942673832971
  num_evaluated: 46816


In [ ]:
try: del trained_poincare_d50
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Poincaré d=50 (RSGD + permanent ROOT freeze + hierarchy)")
print("=" * 60)
trained_poincare_d50, loss_history_poincare_d50 = train_poincare_with_freeze_and_hierarchy(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    dim=50,
    num_epochs=300, burnin_epochs=10, freeze_train_epochs=10000,  # permanent
    learning_rate=10.0, margin=0.05, lambda_h=1.0,
    device=device,
)

print("\nEvaluating Poincaré d=50...")
metrics_poincare_d50 = evaluate_reconstruction(
    trained_poincare_d50, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_poincare_d50.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_poincare_d50.state_dict(),
    'method': 'poincare_K-1_permanent_freeze_hier', 'dim': 50,
    'config': {
        'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300,
        'num_negatives': 50, 'batch_size': 1024,
    },
    'metrics': metrics_poincare_d50,
    'final_loss': loss_history_poincare_d50[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'poincare_d50_permanent_freeze_hier.pt'))

sweep_results['poincare_d50'] = metrics_poincare_d50

Poincaré d=50 (RSGD + permanent ROOT freeze + hierarchy)
🔒+🏛️ ROOT-FREEZE + HIERARCHY Poincaré training
   ROOT idx = 27818, pinned for 10010 epochs
   hierarchy: margin=0.05, lambda_h=1.0
   lr=10.0, num_negatives=50
After init: ROOT_norm = 0.000000   (must be 0.000000)


Epoch   1/300 [burn-in+freeze, lr=0.1000]: loss=3.9818, ROOT_norm=0.0000, mean_norm=0.0041, max_norm=0.0051


Epoch   2/300 [burn-in+freeze, lr=0.1000]: loss=3.9816, ROOT_norm=0.0000, mean_norm=0.0041, max_norm=0.0052


Epoch   3/300 [burn-in+freeze, lr=0.1000]: loss=3.9814, ROOT_norm=0.0000, mean_norm=0.0041, max_norm=0.0052


Epoch  10/300 [burn-in+freeze, lr=0.1000]: loss=3.9800, ROOT_norm=0.0000, mean_norm=0.0041, max_norm=0.0054


...

Epoch 280/300 [FROZEN+hier, lr=10.0000]: loss=0.7919, ROOT_norm=0.0000, mean_norm=0.8828, max_norm=0.9024


Epoch 290/300 [FROZEN+hier, lr=10.0000]: loss=0.7697, ROOT_norm=0.0000, mean_norm=0.8877, max_norm=0.9070


Epoch 300/300 [FROZEN+hier, lr=10.0000]: loss=0.7503, ROOT_norm=0.0000, mean_norm=0.8921, max_norm=0.9109

Evaluating Poincaré d=50...


Evaluating: 100%|██████████| 183/183 [00:35<00:00,  5.17it/s]


  mean_rank: 59.348534688995215
  median_rank: 3.0
  MAP: 0.8186983810126992
  mrr: 0.4211172889890217
  num_evaluated: 46816


In [ ]:
# Add the d=10 results we already have (from earlier in the session)
sweep_results['euclidean_d10'] = {
    'mean_rank': 7.71, 'median_rank': 3.0,
    'MAP': 0.7773, 'mrr': 0.4196, 'num_evaluated': 46816,
}
sweep_results['poincare_d10'] = {
    'mean_rank': 497.3826, 'median_rank': 3.0,
    'MAP': 0.7399, 'mrr': 0.4081, 'num_evaluated': 46816,
}

print("\n" + "=" * 85)
print("DIMENSION SWEEP RESULTS")
print("=" * 85)
print(f"\n{'Method':<35s} | {'Dim':>4} | {'MAP':>7} | {'MRR':>7} | {'mean_rank':>10} | {'median':>7}")
print("-" * 85)

ordered_keys = [
    ('euclidean_d5', 'Euclidean', 5),
    ('poincare_d5',  'Poincaré K=-1 (freeze+hier)', 5),
    ('euclidean_d10', 'Euclidean', 10),
    ('poincare_d10',  'Poincaré K=-1 (freeze+hier)', 10),
    ('euclidean_d50', 'Euclidean', 50),
    ('poincare_d50',  'Poincaré K=-1 (freeze+hier)', 50),
]

for key, label, d in ordered_keys:
    if key not in sweep_results:
        print(f"{label:<35s} | {d:>4} |   ----- |   ----- |     ----- |   -----")
        continue
    m = sweep_results[key]
    print(f"{label:<35s} | {d:>4} | "
          f"{m['MAP']:.4f} | {m['mrr']:.4f} | "
          f"{m['mean_rank']:>10.2f} | {m['median_rank']:>7.1f}")

# Compute Poincaré - Euclidean MAP gap at each dimension
print("\nMAP gap (Poincaré − Euclidean):")
for d in [5, 10, 50]:
    e_key, p_key = f'euclidean_d{d}', f'poincare_d{d}'
    if e_key in sweep_results and p_key in sweep_results:
        gap = sweep_results[p_key]['MAP'] - sweep_results[e_key]['MAP']
        winner = "Poincaré wins" if gap > 0 else "Euclidean wins"
        print(f"  d={d:<3d}: {gap:+.4f}   ({winner})")

# Save sweep results to a JSON
sweep_log_path = os.path.join(RESULTS_DIR, 'dimension_sweep_d5_d10_d50.json')
with open(sweep_log_path, 'w') as f:
    json.dump({
        'date_compiled': datetime.now().isoformat(),
        'results': sweep_results,
    }, f, indent=2)
print(f"\nSaved sweep results to {sweep_log_path}")


DIMENSION SWEEP RESULTS

Method                              |  Dim |     MAP |     MRR |  mean_rank |  median
-------------------------------------------------------------------------------------
Euclidean                           |    5 | 0.2727 | 0.1839 |      88.81 |    17.0
Poincaré K=-1 (freeze+hier)         |    5 | 0.6827 | 0.3919 |     989.37 |     3.0
Euclidean                           |   10 | 0.7773 | 0.4196 |       7.71 |     3.0
Poincaré K=-1 (freeze+hier)         |   10 | 0.7399 | 0.4081 |     497.38 |     3.0
Euclidean                           |   50 | 0.9001 | 0.4170 |     109.81 |     3.0
Poincaré K=-1 (freeze+hier)         |   50 | 0.8187 | 0.4211 |      59.35 |     3.0

MAP gap (Poincaré − Euclidean):
  d=5  : +0.4100   (Poincaré wins)
  d=10 : -0.0374   (Euclidean wins)
  d=50 : -0.0814   (Euclidean wins)

Saved sweep results to /content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/resul

# Baseline 3: Learnable K (d=10) — TODO

In [ ]:
import torch.nn.functional as F


class PoincareEmbeddingLearnableK(nn.Module):
    """
    Poincaré embedding with a learnable global curvature parameter c.

    c is parameterized as softplus(c_raw) to enforce c > 0, then clamped
    to [c_min, c_max] to prevent the c-collapse degeneracy that occurred
    with the unbounded version (c → 0 caused manifold/distance inconsistency
    because the geoopt ManifoldParameter constraint is fixed at init_c=1.0).
    """
    def __init__(self, num_nodes, dim, init_scale=0.001, init_c=1.0,
                 c_min=0.1, c_max=10.0):
        super().__init__()
        # softplus⁻¹(1.0) ≈ 0.5413
        c_raw_init = float(torch.log(torch.expm1(torch.tensor(init_c, dtype=torch.float32))))
        self.c_raw = nn.Parameter(torch.tensor(c_raw_init, dtype=torch.float32))

        # Clamp bounds — keep c in a range that's roughly consistent with the
        # manifold's fixed c=1.0 boundary constraint
        self.c_min = c_min
        self.c_max = c_max

        self.manifold = geoopt.PoincareBall(c=init_c)
        self.embeddings = geoopt.ManifoldParameter(
            torch.empty(num_nodes, dim),
            manifold=self.manifold,
        )
        with torch.no_grad():
            self.embeddings.data.uniform_(-init_scale, init_scale)
        self.dim = dim
        self.num_nodes = num_nodes

    @property
    def c(self):
        """Current curvature magnitude. Clamped to [c_min, c_max]."""
        return F.softplus(self.c_raw).clamp(min=self.c_min, max=self.c_max)

    def forward(self, indices):
        return self.embeddings[indices]

    def distance(self, u, v):
        c = self.c
        sqrt_c = torch.sqrt(c)
        diff_norm_sq = torch.sum((u - v) ** 2, dim=-1)
        u_norm_sq = torch.sum(u ** 2, dim=-1)
        v_norm_sq = torch.sum(v ** 2, dim=-1)
        denom = (1.0 - c * u_norm_sq) * (1.0 - c * v_norm_sq)
        denom = denom.clamp(min=1e-15)
        inner = 1.0 + 2.0 * c * diff_norm_sq / denom
        inner = inner.clamp(min=1.0 + 1e-7)
        return torch.acosh(inner) / sqrt_c

In [ ]:
def train_poincare_learnable_K(
    edges_idx, N, connected, code_to_idx,
    dim=10,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    batch_size=1024,
    num_negatives=50,
    learning_rate=10.0,
    learning_rate_c=0.01,
    burnin_multiplier=0.01,
    init_scale=0.001,
    init_c=1.0,
    margin=0.05,
    lambda_h=1.0,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("=" * 60)
    print(f"📐 LEARNABLE-K Poincaré training (init_c={init_c}, lr_c={learning_rate_c})")
    print(f"   ROOT pinned for {burnin_epochs + freeze_train_epochs} epochs")
    print(f"   hierarchy: margin={margin}, lambda_h={lambda_h}")
    print(f"   lr_emb={learning_rate}, num_negatives={num_negatives}")
    print("=" * 60)

    model = PoincareEmbeddingLearnableK(
        num_nodes=N, dim=dim, init_scale=init_scale, init_c=init_c
    ).to(device)

    root_idx = code_to_idx['ROOT']
    with torch.no_grad():
        model.embeddings.data[root_idx].zero_()

    with torch.no_grad():
        print(f"After init: ROOT_norm = {torch.norm(model.embeddings.data[root_idx]).item():.6f}")
        print(f"            c = {model.c.item():.4f}   (should be ~{init_c})")

    embedding_params = [model.embeddings]
    c_params = [model.c_raw]
    optimizer = geoopt.optim.RiemannianSGD(
        [
            {'params': embedding_params, 'lr': learning_rate},
            {'params': c_params, 'lr': learning_rate_c},
        ],
        lr=learning_rate,
    )

    loss_history = []
    c_history = []
    freeze_until_epoch = burnin_epochs + freeze_train_epochs

    for epoch in range(num_epochs):
        embedding_lr = (learning_rate * burnin_multiplier
                        if epoch < burnin_epochs else learning_rate)
        optimizer.param_groups[0]['lr'] = embedding_lr

        c_lr = 0.0 if epoch < burnin_epochs else learning_rate_c
        optimizer.param_groups[1]['lr'] = c_lr

        freeze_indices = [root_idx] if epoch < freeze_until_epoch else None

        avg_loss = train_one_epoch_with_freeze_and_hierarchy(
            model, edges_idx, optimizer,
            batch_size, num_negatives, N, connected, device,
            margin=margin, lambda_h=lambda_h,
            freeze_indices=freeze_indices,
        )
        loss_history.append(avg_loss)
        c_history.append(model.c.item())

        if epoch < burnin_epochs:
            phase = "burn-in"
        elif epoch < freeze_until_epoch:
            phase = "FROZEN+hier+K"
        else:
            phase = "free+hier+K"

        if (epoch < 3 or epoch == burnin_epochs or (epoch + 1) % 20 == 0
                or epoch == num_epochs - 1):
            with torch.no_grad():
                norms = torch.norm(model.embeddings.data, dim=-1)
                print(
                    f"Epoch {epoch+1:3d}/{num_epochs} [{phase}, lr_e={embedding_lr:.4f}]: "
                    f"loss={avg_loss:.4f}, c={model.c.item():.4f}, "
                    f"mean_norm={norms.mean().item():.4f}, "
                    f"max_norm={norms.max().item():.4f}"
                )

    return model, loss_history, c_history

##Learnable K at d=5

In [ ]:
import gc

if 'learnable_K_results' not in globals():
    learnable_K_results = {}

try: del trained_poincare_learnable_K_d5
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("#" * 70)
print("# Poincaré Learnable K, d=5")
print("#" * 70)

trained_poincare_learnable_K_d5, loss_history_lk_d5, c_history_lk_d5 = train_poincare_learnable_K(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    dim=5,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    learning_rate=10.0,
    learning_rate_c=0.01,
    init_c=1.0,
    margin=0.05,
    lambda_h=1.0,
    device=device,
)

print(f"\nEvaluating Poincaré Learnable K d=5...")
metrics_lk_d5 = evaluate_reconstruction(
    trained_poincare_learnable_K_d5, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_lk_d5.items():
    print(f"  {k}: {v}")

final_c_d5 = c_history_lk_d5[-1]
print(f"\nFinal learned c: {final_c_d5:.4f}  (initial: 1.0)")

torch.save({
    'model_state_dict': trained_poincare_learnable_K_d5.state_dict(),
    'method': 'poincare_learnable_K',
    'dim': 5,
    'config': {
        'optimizer': 'RiemannianSGD',
        'lr': 10.0, 'lr_c': 0.01, 'init_c': 1.0,
        'c_min': 0.1, 'c_max': 10.0,
        'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0,
        'epochs': 300, 'num_negatives': 50, 'batch_size': 1024,
    },
    'metrics': metrics_lk_d5,
    'final_loss': loss_history_lk_d5[-1],
    'final_c': final_c_d5,
    'loss_history': loss_history_lk_d5,
    'c_history': c_history_lk_d5,
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'poincare_learnable_K_d5.pt'))

learnable_K_results['d5'] = {
    'metrics': metrics_lk_d5,
    'final_c': final_c_d5,
    'final_loss': loss_history_lk_d5[-1],
}
print(f"\nSaved. d=5 done.")

######################################################################
# Poincaré Learnable K, d=5
######################################################################
📐 LEARNABLE-K Poincaré training (init_c=1.0, lr_c=0.01)
   ROOT pinned for 10010 epochs
   hierarchy: margin=0.05, lambda_h=1.0
   lr_emb=10.0, num_negatives=50
After init: ROOT_norm = 0.000000
            c = 1.0000   (should be ~1.0)


Epoch   1/300 [burn-in, lr_e=0.1000]: loss=3.9818, c=1.0000, mean_norm=0.0013, max_norm=0.0021


Epoch   2/300 [burn-in, lr_e=0.1000]: loss=3.9816, c=1.0000, mean_norm=0.0013, max_norm=0.0022


Epoch   3/300 [burn-in, lr_e=0.1000]: loss=3.9814, c=1.0000, mean_norm=0.0013, max_norm=0.0022


Epoch  11/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.9813, c=1.0000, mean_norm=0.0059, max_norm=0.0170


...

Epoch 260/300 [FROZEN+hier+K, lr_e=10.0000]: loss=1.4538, c=1.6742, mean_norm=0.8224, max_norm=0.9960


Epoch 280/300 [FROZEN+hier+K, lr_e=10.0000]: loss=1.4128, c=0.9903, mean_norm=0.8341, max_norm=0.9960


Epoch 300/300 [FROZEN+hier+K, lr_e=10.0000]: loss=1.2472, c=0.9906, mean_norm=0.8585, max_norm=0.9960

Evaluating Poincaré Learnable K d=5...


Evaluating: 100%|██████████| 183/183 [02:13<00:00,  1.37it/s]


  mean_rank: 2713.8345223855094
  median_rank: 3.0
  MAP: 0.6543730869238694
  mrr: 0.38086857723821504
  num_evaluated: 46816

Final learned c: 0.9906  (initial: 1.0)

Saved. d=5 done.


##Learnable K at d=10

In [ ]:
try: del trained_poincare_learnable_K_d10
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("#" * 70)
print("# Poincaré Learnable K, d=10")
print("#" * 70)

trained_poincare_learnable_K_d10, loss_history_lk_d10, c_history_lk_d10 = train_poincare_learnable_K(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    dim=10,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    learning_rate=10.0,
    learning_rate_c=0.01,
    init_c=1.0,
    margin=0.05,
    lambda_h=1.0,
    device=device,
)

print(f"\nEvaluating Poincaré Learnable K d=10...")
metrics_lk_d10 = evaluate_reconstruction(
    trained_poincare_learnable_K_d10, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_lk_d10.items():
    print(f"  {k}: {v}")

final_c_d10 = c_history_lk_d10[-1]
print(f"\nFinal learned c: {final_c_d10:.4f}  (initial: 1.0)")

torch.save({
    'model_state_dict': trained_poincare_learnable_K_d10.state_dict(),
    'method': 'poincare_learnable_K',
    'dim': 10,
    'config': {
        'optimizer': 'RiemannianSGD',
        'lr': 10.0, 'lr_c': 0.01, 'init_c': 1.0,
        'c_min': 0.1, 'c_max': 10.0,
        'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0,
        'epochs': 300, 'num_negatives': 50, 'batch_size': 1024,
    },
    'metrics': metrics_lk_d10,
    'final_loss': loss_history_lk_d10[-1],
    'final_c': final_c_d10,
    'loss_history': loss_history_lk_d10,
    'c_history': c_history_lk_d10,
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'poincare_learnable_K_d10.pt'))

learnable_K_results['d10'] = {
    'metrics': metrics_lk_d10,
    'final_c': final_c_d10,
    'final_loss': loss_history_lk_d10[-1],
}
print(f"\nSaved. d=10 done.")

######################################################################
# Poincaré Learnable K, d=10
######################################################################
📐 LEARNABLE-K Poincaré training (init_c=1.0, lr_c=0.01)
   ROOT pinned for 10010 epochs
   hierarchy: margin=0.05, lambda_h=1.0
   lr_emb=10.0, num_negatives=50
After init: ROOT_norm = 0.000000
            c = 1.0000   (should be ~1.0)


Epoch   1/300 [burn-in, lr_e=0.1000]: loss=3.9818, c=1.0000, mean_norm=0.0018, max_norm=0.0027


Epoch   2/300 [burn-in, lr_e=0.1000]: loss=3.9816, c=1.0000, mean_norm=0.0018, max_norm=0.0028


Epoch   3/300 [burn-in, lr_e=0.1000]: loss=3.9814, c=1.0000, mean_norm=0.0018, max_norm=0.0028


Epoch  11/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.9812, c=1.0000, mean_norm=0.0062, max_norm=0.0129


...

Epoch 260/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.3358, c=6.0444, mean_norm=0.7293, max_norm=0.9960


Epoch 280/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.0271, c=4.4386, mean_norm=0.7125, max_norm=0.9960


Epoch 300/300 [FROZEN+hier+K, lr_e=10.0000]: loss=2.5444, c=2.7606, mean_norm=0.6881, max_norm=0.9960

Evaluating Poincaré Learnable K d=10...


Evaluating: 100%|██████████| 183/183 [01:39<00:00,  1.84it/s]

  mean_rank: 2698.964734278879
  median_rank: 4.0
  MAP: 0.554977898033975
  mrr: 0.32390910149560936
  num_evaluated: 46816

Final learned c: 2.7606  (initial: 1.0)

Saved. d=10 done.


##Learnable K at d=50

In [ ]:
try: del trained_poincare_learnable_K_d50
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("#" * 70)
print("# Poincaré Learnable K, d=50")
print("#" * 70)

trained_poincare_learnable_K_d50, loss_history_lk_d50, c_history_lk_d50 = train_poincare_learnable_K(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    dim=50,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    learning_rate=10.0,
    learning_rate_c=0.01,
    init_c=1.0,
    margin=0.05,
    lambda_h=1.0,
    device=device,
)

print(f"\nEvaluating Poincaré Learnable K d=50...")
metrics_lk_d50 = evaluate_reconstruction(
    trained_poincare_learnable_K_d50, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_lk_d50.items():
    print(f"  {k}: {v}")

final_c_d50 = c_history_lk_d50[-1]
print(f"\nFinal learned c: {final_c_d50:.4f}  (initial: 1.0)")

torch.save({
    'model_state_dict': trained_poincare_learnable_K_d50.state_dict(),
    'method': 'poincare_learnable_K',
    'dim': 50,
    'config': {
        'optimizer': 'RiemannianSGD',
        'lr': 10.0, 'lr_c': 0.01, 'init_c': 1.0,
        'c_min': 0.1, 'c_max': 10.0,
        'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0,
        'epochs': 300, 'num_negatives': 50, 'batch_size': 1024,
    },
    'metrics': metrics_lk_d50,
    'final_loss': loss_history_lk_d50[-1],
    'final_c': final_c_d50,
    'loss_history': loss_history_lk_d50,
    'c_history': c_history_lk_d50,
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'poincare_learnable_K_d50.pt'))

learnable_K_results['d50'] = {
    'metrics': metrics_lk_d50,
    'final_c': final_c_d50,
    'final_loss': loss_history_lk_d50[-1],
}
print(f"\nSaved. d=50 done.")

######################################################################
# Poincaré Learnable K, d=50
######################################################################
📐 LEARNABLE-K Poincaré training (init_c=1.0, lr_c=0.01)
   ROOT pinned for 10010 epochs
   hierarchy: margin=0.05, lambda_h=1.0
   lr_emb=10.0, num_negatives=50
After init: ROOT_norm = 0.000000
            c = 1.0000   (should be ~1.0)


Epoch   1/300 [burn-in, lr_e=0.1000]: loss=3.9818, c=1.0000, mean_norm=0.0041, max_norm=0.0051


Epoch   2/300 [burn-in, lr_e=0.1000]: loss=3.9816, c=1.0000, mean_norm=0.0041, max_norm=0.0051


Epoch   3/300 [burn-in, lr_e=0.1000]: loss=3.9814, c=1.0000, mean_norm=0.0041, max_norm=0.0051


Epoch  11/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.9805, c=1.0000, mean_norm=0.0074, max_norm=0.0114


...

Epoch 260/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.8224, c=6.6691, mean_norm=0.6323, max_norm=0.9960


Epoch 280/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.8523, c=6.6240, mean_norm=0.6543, max_norm=0.9960


Epoch 300/300 [FROZEN+hier+K, lr_e=10.0000]: loss=3.8576, c=6.3421, mean_norm=0.6734, max_norm=0.9960

Evaluating Poincaré Learnable K d=50...


Evaluating: 100%|██████████| 183/183 [02:15<00:00,  1.35it/s]

  mean_rank: 4712.958924299385
  median_rank: 6.0
  MAP: 0.5552850854175266
  mrr: 0.2950007589637548
  num_evaluated: 46816

Final learned c: 6.3421  (initial: 1.0)

Saved. d=50 done.


In [ ]:
import json, os

sweep_path = os.path.join(RESULTS_DIR, 'learnable_K_sweep.json')

sweep = {}
if os.path.exists(sweep_path):
    with open(sweep_path, 'r') as f:
        sweep = json.load(f)

sweep['d50'] = {
    'mean_rank': 4712.958924299385,
    'median_rank': 6.0,
    'MAP': 0.5552850854175266,
    'MRR': 0.2950007589637548,
    'num_evaluated': 46816,
    'final_c': 6.3421,
    'initial_c': 1.0,
    'notes': 'consistent with known projection-mismatch instability; c drift scales monotonically with dim (d5: ~1.0, d10: 2.76, d50: 6.34); long-tail boundary-pinned failure (median_rank=6 vs mean_rank=4713 → handful of catastrophically misplaced nodes, not global collapse); MAP plateaus across d=10/50 because broken-gradient regime caps performance independent of capacity',
}

with open(sweep_path, 'w') as f:
    json.dump(sweep, f, indent=2)

print(f"Logged. Sweep now contains keys: {list(sweep.keys())}")

Logged. Sweep now contains keys: ['d50']


# Baseline 4: Mixed-Curvature Product Spaces — TODO

In [20]:
import geoopt

class MixedCurvatureEmbedding(nn.Module):
    """
    Mixed-curvature embedding on a product manifold:
        M = PoincareBall(c=1.0)^{dim_h} × Euclidean^{dim_e}

    Following Gu et al. 2019: split dimensions between a hyperbolic part
    (captures hierarchy) and a Euclidean part (captures non-hierarchical
    structure). Product distance is Pythagorean: sqrt(d_H^2 + d_E^2).
    """
    def __init__(self, num_nodes, dim_h, dim_e, init_scale=0.001, c=1.0):
        super().__init__()
        self.num_nodes = num_nodes
        self.dim_h = dim_h
        self.dim_e = dim_e
        self.total_dim = dim_h + dim_e

        # ProductManifold takes (manifold, dim) pairs; combined tensor layout is
        # [hyperbolic coords (dim_h), euclidean coords (dim_e)] along last axis.
        self.manifold = geoopt.ProductManifold(
            (geoopt.PoincareBall(c=c), dim_h),
            (geoopt.Euclidean(), dim_e),
        )

        # Small random init, then projx to ensure on-manifold (mostly matters for the
        # Poincaré half: enforces ‖x_h‖ < 1/sqrt(c)). At init_scale=0.001 this is safe.
        init = torch.randn(num_nodes, self.total_dim) * init_scale
        init = self.manifold.projx(init)
        self.embeddings = geoopt.ManifoldParameter(init, manifold=self.manifold)

    def forward(self, indices):
        # indices: (batch,) long → returns (batch, total_dim)
        return self.embeddings[indices]

    def distance(self, u, v):
        # u, v: (..., total_dim)  →  (...,)
        # Product-manifold geodesic distance, sqrt(d_H^2 + d_E^2).
        return self.manifold.dist(u, v)

    def hyperbolic_norm(self, x):
        # x: (..., total_dim)  →  (...,)
        # Euclidean norm of just the hyperbolic coords. Monotonic with hyperbolic
        # distance from origin; used for hierarchy regularization.
        return torch.norm(x[..., :self.dim_h], p=2, dim=-1)


def compute_loss_with_hierarchy_mixed(
    anchor_emb, positive_emb, negative_embs, distance_fn, hyperbolic_norm_fn,
    margin=0.05, lambda_h=1.0,
):
    """NK softmax + hierarchy reg restricted to the hyperbolic component."""
    pos_dist = distance_fn(anchor_emb, positive_emb)                       # (B,)
    neg_dist = distance_fn(anchor_emb.unsqueeze(1), negative_embs)         # (B, K)
    all_dist = torch.cat([pos_dist.unsqueeze(1), neg_dist], dim=1)         # (B, 1+K)
    logsumexp_term = torch.logsumexp(-all_dist, dim=1)                     # (B,)
    nk_per_anchor = pos_dist + logsumexp_term                              # (B,)

    # Hierarchy: parent (anchor) should be closer to origin than child (positive)
    anchor_h = hyperbolic_norm_fn(anchor_emb)
    positive_h = hyperbolic_norm_fn(positive_emb)
    hierarchy_violation = torch.relu(anchor_h - positive_h + margin)

    return nk_per_anchor.mean() + lambda_h * hierarchy_violation.mean()


def train_one_epoch_mixed(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0, freeze_indices=None,
):
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)           # (B, D_total)
        positive_emb = model(positive_t)       # (B, D_total)
        negative_embs = model(negative_t)      # (B, K, D_total)

        loss = compute_loss_with_hierarchy_mixed(
            anchor_emb, positive_emb, negative_embs,
            model.distance, model.hyperbolic_norm,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)


def train_mixed_curvature(
    edges_idx, N, connected, code_to_idx,
    total_dim=10,
    num_epochs=300, burnin_epochs=10, freeze_train_epochs=10000,
    batch_size=1024, num_negatives=50,
    learning_rate=10.0, burnin_multiplier=0.01, init_scale=0.001,
    margin=0.05, lambda_h=1.0, c=1.0,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Split total_dim — hyperbolic gets the larger half when odd
    dim_h = total_dim // 2 + (total_dim % 2)
    dim_e = total_dim // 2

    print("=" * 60)
    print(f"🔀 MIXED-CURVATURE training (Poincaré^{dim_h} × R^{dim_e})")
    print(f"   ROOT idx = {code_to_idx['ROOT']}, pinned for {burnin_epochs + freeze_train_epochs} epochs")
    print(f"   hierarchy on H component only: margin={margin}, lambda_h={lambda_h}")
    print(f"   lr={learning_rate}, num_negatives={num_negatives}")
    print("=" * 60)

    model = MixedCurvatureEmbedding(
        num_nodes=N, dim_h=dim_h, dim_e=dim_e,
        init_scale=init_scale, c=c,
    ).to(device)

    root_idx = code_to_idx['ROOT']
    with torch.no_grad():
        model.embeddings.data[root_idx].zero_()
        root_norm = torch.norm(model.embeddings.data[root_idx]).item()
        print(f"After init: ROOT_norm = {root_norm:.6f}   (must be 0.000000)")

    optimizer = geoopt.optim.RiemannianSGD(model.parameters(), lr=learning_rate)

    loss_history = []
    freeze_until_epoch = burnin_epochs + freeze_train_epochs

    for epoch in range(num_epochs):
        effective_lr = (learning_rate * burnin_multiplier
                        if epoch < burnin_epochs else learning_rate)
        for pg in optimizer.param_groups:
            pg['lr'] = effective_lr

        freeze_indices = [root_idx] if epoch < freeze_until_epoch else None

        avg_loss = train_one_epoch_mixed(
            model, edges_idx, optimizer,
            batch_size, num_negatives, N, connected, device,
            margin=margin, lambda_h=lambda_h,
            freeze_indices=freeze_indices,
        )
        loss_history.append(avg_loss)

        phase = ("burn-in+freeze" if epoch < burnin_epochs
                 else "FROZEN+hier" if epoch < freeze_until_epoch
                 else "free+hier")

        if (epoch < 3 or epoch == burnin_epochs or epoch == freeze_until_epoch
                or (epoch + 1) % 10 == 0):
            with torch.no_grad():
                h_part = model.embeddings.data[..., :dim_h]
                e_part = model.embeddings.data[..., dim_h:]
                h_norms = torch.norm(h_part, dim=-1)
                e_norms = torch.norm(e_part, dim=-1)
                print(
                    f"Epoch {epoch+1:3d}/{num_epochs} [{phase}, lr={effective_lr:.4f}]: "
                    f"loss={avg_loss:.4f}, "
                    f"ROOT_h={h_norms[root_idx].item():.4f}, "
                    f"mean_h={h_norms.mean().item():.4f}, "
                    f"max_h={h_norms.max().item():.4f}, "
                    f"mean_e={e_norms.mean().item():.4f}"
                )

    return model, loss_history

In [23]:
try: del trained_mixed_d5
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Mixed-Curvature d=5 (Poincaré^3 × R^2, RSGD + ROOT freeze + H-hierarchy)")
print("=" * 60)
trained_mixed_d5, loss_history_mixed_d5 = train_mixed_curvature(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    total_dim=5,
    num_epochs=300, burnin_epochs=10, freeze_train_epochs=10000,
    learning_rate=10.0, margin=0.05, lambda_h=1.0,
    device=device,
)

print("\nEvaluating Mixed-Curvature d=5...")
metrics_mixed_d5 = evaluate_reconstruction(
    trained_mixed_d5, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_mixed_d5.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_mixed_d5.state_dict(),
    'method': 'mixed_curvature_PxE_permanent_freeze_hier', 'dim': 5,
    'config': {
        'total_dim': 5, 'dim_h': 3, 'dim_e': 2,
        'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300,
        'num_negatives': 50, 'batch_size': 1024, 'c': 1.0,
    },
    'metrics': metrics_mixed_d5,
    'final_loss': loss_history_mixed_d5[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'mixed_curvature_d5.pt'))

sweep_results['mixed_curvature_d5'] = metrics_mixed_d5

Mixed-Curvature d=5 (Poincaré^3 × R^2, RSGD + ROOT freeze + H-hierarchy)
🔀 MIXED-CURVATURE training (Poincaré^3 × R^2)
   ROOT idx = 27818, pinned for 10010 epochs
   hierarchy on H component only: margin=0.05, lambda_h=1.0
   lr=10.0, num_negatives=50
After init: ROOT_norm = 0.000000   (must be 0.000000)


Epoch   1/300 [burn-in+freeze, lr=0.1000]: loss=3.9818, ROOT_h=0.0000, mean_h=0.0016, max_h=0.0051, mean_e=0.0013


Epoch   2/300 [burn-in+freeze, lr=0.1000]: loss=3.9816, ROOT_h=0.0000, mean_h=0.0016, max_h=0.0051, mean_e=0.0013


Epoch   3/300 [burn-in+freeze, lr=0.1000]: loss=3.9814, ROOT_h=0.0000, mean_h=0.0016, max_h=0.0051, mean_e=0.0013


Epoch  10/300 [burn-in+freeze, lr=0.1000]: loss=3.9803, ROOT_h=0.0000, mean_h=0.0017, max_h=0.0053, mean_e=0.0013


...

Epoch 280/300 [FROZEN+hier, lr=10.0000]: loss=1.2931, ROOT_h=0.0000, mean_h=0.6927, max_h=0.8867, mean_e=1.6419


Epoch 290/300 [FROZEN+hier, lr=10.0000]: loss=1.2388, ROOT_h=0.0000, mean_h=0.7066, max_h=0.8934, mean_e=1.6754


Epoch 300/300 [FROZEN+hier, lr=10.0000]: loss=1.1901, ROOT_h=0.0000, mean_h=0.7196, max_h=0.8999, mean_e=1.7074

Evaluating Mixed-Curvature d=5...


Evaluating: 100%|██████████| 183/183 [01:04<00:00,  2.82it/s]

  mean_rank: 746.0102956254271
  median_rank: 3.0
  MAP: 0.6806030706135352
  mrr: 0.39506160325435413
  num_evaluated: 46816


In [24]:
try: del trained_mixed_d10
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Mixed-Curvature d=10 (Poincaré^5 × R^5, RSGD + ROOT freeze + H-hierarchy)")
print("=" * 60)
trained_mixed_d10, loss_history_mixed_d10 = train_mixed_curvature(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    total_dim=10,
    num_epochs=300, burnin_epochs=10, freeze_train_epochs=10000,
    learning_rate=10.0, margin=0.05, lambda_h=1.0,
    device=device,
)

print("\nEvaluating Mixed-Curvature d=10...")
metrics_mixed_d10 = evaluate_reconstruction(
    trained_mixed_d10, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_mixed_d10.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_mixed_d10.state_dict(),
    'method': 'mixed_curvature_PxE_permanent_freeze_hier', 'dim': 10,
    'config': {
        'total_dim': 10, 'dim_h': 5, 'dim_e': 5,
        'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300,
        'num_negatives': 50, 'batch_size': 1024, 'c': 1.0,
    },
    'metrics': metrics_mixed_d10,
    'final_loss': loss_history_mixed_d10[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'mixed_curvature_d10.pt'))

sweep_results['mixed_curvature_d10'] = metrics_mixed_d10

Mixed-Curvature d=10 (Poincaré^5 × R^5, RSGD + ROOT freeze + H-hierarchy)
🔀 MIXED-CURVATURE training (Poincaré^5 × R^5)
   ROOT idx = 27818, pinned for 10010 epochs
   hierarchy on H component only: margin=0.05, lambda_h=1.0
   lr=10.0, num_negatives=50
After init: ROOT_norm = 0.000000   (must be 0.000000)


Epoch   1/300 [burn-in+freeze, lr=0.1000]: loss=3.9818, ROOT_h=0.0000, mean_h=0.0021, max_h=0.0053, mean_e=0.0021


Epoch   2/300 [burn-in+freeze, lr=0.1000]: loss=3.9816, ROOT_h=0.0000, mean_h=0.0021, max_h=0.0053, mean_e=0.0021


Epoch   3/300 [burn-in+freeze, lr=0.1000]: loss=3.9814, ROOT_h=0.0000, mean_h=0.0021, max_h=0.0053, mean_e=0.0021


Epoch  10/300 [burn-in+freeze, lr=0.1000]: loss=3.9801, ROOT_h=0.0000, mean_h=0.0022, max_h=0.0055, mean_e=0.0022


...

Epoch 280/300 [FROZEN+hier, lr=10.0000]: loss=1.0746, ROOT_h=0.0000, mean_h=0.5777, max_h=0.8620, mean_e=2.2303


Epoch 290/300 [FROZEN+hier, lr=10.0000]: loss=1.0255, ROOT_h=0.0000, mean_h=0.5910, max_h=0.8696, mean_e=2.2800


Epoch 300/300 [FROZEN+hier, lr=10.0000]: loss=0.9797, ROOT_h=0.0000, mean_h=0.6036, max_h=0.8764, mean_e=2.3275

Evaluating Mixed-Curvature d=10...


Evaluating: 100%|██████████| 183/183 [00:46<00:00,  3.91it/s]

  mean_rank: 243.82587149692412
  median_rank: 3.0
  MAP: 0.756310004497069
  mrr: 0.42016444305216005
  num_evaluated: 46816


In [25]:
try: del trained_mixed_d50
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

print("=" * 60)
print("Mixed-Curvature d=50 (Poincaré^25 × R^25, RSGD + ROOT freeze + H-hierarchy)")
print("=" * 60)
trained_mixed_d50, loss_history_mixed_d50 = train_mixed_curvature(
    edges_idx=edges_idx, N=N, connected=connected, code_to_idx=code_to_idx,
    total_dim=50,
    num_epochs=300, burnin_epochs=10, freeze_train_epochs=10000,
    learning_rate=10.0, margin=0.05, lambda_h=1.0,
    device=device,
)

print("\nEvaluating Mixed-Curvature d=50...")
metrics_mixed_d50 = evaluate_reconstruction(
    trained_mixed_d50, edges_idx, N, device,
    sample_size=None, batch_size=256,
)
for k, v in metrics_mixed_d50.items():
    print(f"  {k}: {v}")

torch.save({
    'model_state_dict': trained_mixed_d50.state_dict(),
    'method': 'mixed_curvature_PxE_permanent_freeze_hier', 'dim': 50,
    'config': {
        'total_dim': 50, 'dim_h': 25, 'dim_e': 25,
        'lr': 10.0, 'burnin_epochs': 10, 'freeze_train_epochs': 10000,
        'margin': 0.05, 'lambda_h': 1.0, 'epochs': 300,
        'num_negatives': 50, 'batch_size': 1024, 'c': 1.0,
    },
    'metrics': metrics_mixed_d50,
    'final_loss': loss_history_mixed_d50[-1],
    'date_trained': datetime.now().isoformat(),
}, os.path.join(MODELS_DIR, 'mixed_curvature_d50.pt'))

sweep_results['mixed_curvature_d50'] = metrics_mixed_d50

Mixed-Curvature d=50 (Poincaré^25 × R^25, RSGD + ROOT freeze + H-hierarchy)
🔀 MIXED-CURVATURE training (Poincaré^25 × R^25)
   ROOT idx = 27818, pinned for 10010 epochs
   hierarchy on H component only: margin=0.05, lambda_h=1.0
   lr=10.0, num_negatives=50
After init: ROOT_norm = 0.000000   (must be 0.000000)


Epoch   1/300 [burn-in+freeze, lr=0.1000]: loss=3.9818, ROOT_h=0.0000, mean_h=0.0050, max_h=0.0079, mean_e=0.0050


Epoch   2/300 [burn-in+freeze, lr=0.1000]: loss=3.9816, ROOT_h=0.0000, mean_h=0.0050, max_h=0.0079, mean_e=0.0050


Epoch   3/300 [burn-in+freeze, lr=0.1000]: loss=3.9814, ROOT_h=0.0000, mean_h=0.0050, max_h=0.0079, mean_e=0.0050


Epoch  10/300 [burn-in+freeze, lr=0.1000]: loss=3.9799, ROOT_h=0.0000, mean_h=0.0050, max_h=0.0081, mean_e=0.0050


...

Epoch 280/300 [FROZEN+hier, lr=10.0000]: loss=0.8570, ROOT_h=0.0000, mean_h=0.5427, max_h=0.8295, mean_e=2.4817


Epoch 290/300 [FROZEN+hier, lr=10.0000]: loss=0.8107, ROOT_h=0.0000, mean_h=0.5532, max_h=0.8376, mean_e=2.5392


Epoch 300/300 [FROZEN+hier, lr=10.0000]: loss=0.7685, ROOT_h=0.0000, mean_h=0.5630, max_h=0.8449, mean_e=2.5941

Evaluating Mixed-Curvature d=50...


Evaluating: 100%|██████████| 183/183 [00:37<00:00,  4.83it/s]

  mean_rank: 39.690725393028025
  median_rank: 3.0
  MAP: 0.8274095239953723
  mrr: 0.439640074403514
  num_evaluated: 46816


In [31]:
import json, os, torch
from datetime import datetime

def _to_jsonable(v):
    """Convert tensors / numpy scalars / nested structures to JSON-friendly types."""
    if hasattr(v, 'item') and not isinstance(v, (int, float, bool, str)):
        return v.item()
    if isinstance(v, dict):
        return {k: _to_jsonable(vv) for k, vv in v.items()}
    if isinstance(v, (list, tuple)):
        return [_to_jsonable(vv) for vv in v]
    if isinstance(v, (int, float, str, bool)) or v is None:
        return v
    return str(v)

# All 12 runs: (key, method_label, dim, pt_filename, fallback_json or None)
runs_spec = [
    ('euclidean_d5',        'Euclidean',                     5,  'euclidean_d5.pt',                          None),
    ('euclidean_d10',       'Euclidean',                     10, 'euclidean_d10_K50.pt',                     'euclidean_d10_K50_results.json'),
    ('euclidean_d50',       'Euclidean',                     50, 'euclidean_d50.pt',                         None),
    ('poincare_d5',         'Poincaré K=−1',                 5,  'poincare_d5_permanent_freeze_hier.pt',     None),
    ('poincare_d10',        'Poincaré K=−1',                 10, 'poincare_d10_permanent_freeze_hier.pt',    None),
    ('poincare_d50',        'Poincaré K=−1',                 50, 'poincare_d50_permanent_freeze_hier.pt',    None),
    ('learnable_K_d5',      'Learnable K (single global c)', 5,  'poincare_learnable_K_d5.pt',               None),
    ('learnable_K_d10',     'Learnable K (single global c)', 10, 'poincare_learnable_K_d10.pt',              None),
    ('learnable_K_d50',     'Learnable K (single global c)', 50, 'poincare_learnable_K_d50.pt',              None),
    ('mixed_curvature_d5',  'Mixed-Curvature (P × E)',       5,  'mixed_curvature_d5.pt',                    None),
    ('mixed_curvature_d10', 'Mixed-Curvature (P × E)',       10, 'mixed_curvature_d10.pt',                   None),
    ('mixed_curvature_d50', 'Mixed-Curvature (P × E)',       50, 'mixed_curvature_d50.pt',                   None),
]

consolidated = {
    'compiled_at': datetime.now().isoformat(),
    'evaluation': 'reconstruction (full edge set, all 46,816 edges)',
    'num_negatives_train': 50,
    'notes': (
        'Phase 2 baseline sweep — 4 methods × 3 dimensions = 12 runs. '
        'All methods trained with K=50 negatives for fair comparison of the NK softmax loss. '
        'Euclidean: Adam lr=0.1, 50 epochs, no freeze, no hierarchy. '
        'Poincaré / Learnable-K / Mixed-Curvature: RiemannianSGD lr=10.0, '
        'permanent ROOT freeze, hierarchy regularization (margin=0.05, lambda_h=1.0), '
        '300 epochs, 10-epoch burnin at lr*0.01. '
        'NOTE: Euclidean d=10 is the K=50 retrain (5/19); original K=10 version (MAP=0.7773) is superseded.'
    ),
    'runs': {}
}

for key, method, dim, pt_file, json_fallback in runs_spec:
    pt_path = os.path.join(MODELS_DIR, pt_file)
    metrics = None
    config = {}

    if os.path.exists(pt_path):
        try:
            ckpt = torch.load(pt_path, map_location='cpu')
            metrics = ckpt.get('metrics')
            config = ckpt.get('config', {})
        except Exception as e:
            print(f"⚠️  failed to load {pt_path}: {e}")
    else:
        print(f"⚠️  {pt_path} not found")

    # Fallback: standalone results JSON
    if metrics is None and json_fallback is not None:
        json_path = os.path.join(RESULTS_DIR, json_fallback)
        if os.path.exists(json_path):
            with open(json_path) as f:
                metrics = json.load(f)

    if metrics is None:
        print(f"⚠️  SKIPPING {key}: no metrics found")
        continue

    # Normalize MRR capitalization (some saves use 'mrr', some 'MRR')
    metrics = dict(metrics)
    if 'mrr' in metrics and 'MRR' not in metrics:
        metrics['MRR'] = metrics.pop('mrr')

    consolidated['runs'][key] = {
        'method': method,
        'dim': dim,
        'metrics': _to_jsonable(metrics),
        'config': _to_jsonable(config),
    }

out_path = os.path.join(RESULTS_DIR, 'phase2_all_baselines.json')
with open(out_path, 'w') as f:
    json.dump(consolidated, f, indent=2)

print(f"\n{'=' * 90}")
print(f"Consolidated {len(consolidated['runs'])} / {len(runs_spec)} runs → {out_path}")
print('=' * 90)

# Summary table
print(f"\n{'Method':<33} {'Dim':>4} {'MAP':>8} {'MRR':>8} {'mean_rank':>11} {'med_rank':>9}")
print('-' * 78)
for key, run in consolidated['runs'].items():
    m = run['metrics']
    print(f"{run['method']:<33} {run['dim']:>4} "
          f"{m.get('MAP', float('nan')):>8.4f} "
          f"{m.get('MRR', float('nan')):>8.4f} "
          f"{m.get('mean_rank', float('nan')):>11.2f} "
          f"{m.get('median_rank', float('nan')):>9.1f}")


Consolidated 12 / 12 runs → /content/drive/MyDrive/Position-Dependent_Curvature_for_Hyperbolic_Embeddings_of_Medical_Ontologies/hyperbolic_medical_embedding/results/phase2_all_baselines.json

Method                             Dim      MAP      MRR   mean_rank  med_rank
------------------------------------------------------------------------------
Euclidean                            5   0.2727   0.1839       88.81      17.0
Euclidean                           10   0.8606   0.4247        7.07       3.0
Euclidean                           50   0.9001   0.4170      109.81       3.0
Poincaré K=−1                        5   0.6827   0.3919      989.37       3.0
Poincaré K=−1                       10   0.7399   0.4081      497.38       3.0
Poincaré K=−1                       50   0.8187   0.4211       59.35       3.0
Learnable K (single global c)        5   0.6544   0.3809     2713.83       3.0
Learnable K (single global c)       10   0.5550   0.3239     2698.96       4.0
Learnable K (sing